# BattMo.jl Workshop — Setup & Hello World

**Agenda slot:** 9:30 – 10:30

If you haven't already worked through **`Installation_check.ipynb`**, do that first — it makes sure Julia, VSCode and the required packages are correctly installed. This notebook assumes that setup is done, and takes you through your first BattMo.jl simulations.

By the end of this hour you will:
- Understand how BattMo.jl structures its inputs (Parameters and Settings)
- Run your first battery simulation
- Explore and visualize the output
- Make your first parameter change (an assignment, to warm up for the rest of the day)

## Setup check

Let's import BattMo and the plotting packages we'll use throughout the day.

In [70]:
using BattMo, GLMakie, Jutul

## 1 - Input ([docs](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/terminology))

BattMo.jl structures its simulation inputs into two primary categories: Parameters and Settings. This distinction helps users differentiate between the physical characteristics of the battery system and the numerical configurations of the simulation.

**Parameters** represent the controllable variables in real-world experiments. They are further divided into:
- **Cell Parameters**: define the intrinsic properties of the battery cell, such as geometry and material characteristics.
- **Cycling Protocol Parameters**: specify how the cell is operated during a simulation.

**Settings** are used to configure numerical assumptions for solving equations and finding numerical solutions. They are further divided into:
- **Model Settings**: define numerical assumptions related to the battery model, such as diffusion methods or simplifications used in the simulation.
- **Simulation Settings**: define numerical assumptions specific to the simulation.

BattMo stores cell parameters, cycling protocols and settings in a user-friendly JSON format to facilitate reuse. We can load parameters directly from the built-in default sets, which is very convenient for quickly testing a simulation setup. Let's see which default sets BattMo provides.

In [71]:
print_default_input_sets()


📋 Overview of Available Default Sets

📁 cell_parameters:         chayambuka_2022, chen_2020, xu_2015
📁 cycling_protocols:       cc_charge, cc_cycling, cc_discharge, cccv, user_defined_current_function
📁 full_simulation_input:   chen_2020, chen_2020_p4d
📁 model_settings:          p2d, p4d_cylindrical, p4d_pouch
📁 simulation_settings:     p2d, p2d_fine_resolution, p4d_cylindrical, p4d_pouch
📁 solver_settings:         default, direct, iterative

📖 Detailed Descriptions

📂 cell_parameters
----------------------------------------------------------------------------------------------------
chayambuka_2022
🔹 Cell name:       	-
🔹 Cell case:       	Pouch
🔹 Source:          	[visit]((Invalid metadata format))

🔹 Suitable for:
   • RampUp:             Sinusoidal
   • TransportInSolid:   FullDiffusion
   • ModelFramework:     P2D
🔹 Description:     	Parameter set for a Sodium ion cell based on Chayambuka et al. The positive electrode open circuit potential has been retrieved from a [COMSOL examp

For our example, we'll load the cell parameter set of an NMC811 vs. Graphite-SiOx cell whose parameters were determined in the [Chen 2020 paper](https://doi.org/10.1149/1945-7111/ab9050), together with a simple Constant Current Discharge cycling protocol.

In [72]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cycling_protocol = load_cycling_protocol(; from_default_set = "cc_discharge");

This a quick way of testing a setup, but for the purpose of this workshop we would like to be able to see what a parameter set contains. Therefore, we'll retrieve the default parameter sets that BattMo provides and store them locally in a folder. We can do this by running the following script.

In [73]:
path = "."
folder_name = "default_sets"
generate_default_parameter_files(path, folder_name; force = true)

🛠 JSON files successfully written! Path:
	.\default_sets


".\\default_sets"

As we stored the default sets in our own folder, we can alter the default files if we want to and load the parameters from our dedicated folder.

In [74]:
cell_parameters = load_cell_parameters(; from_file_path = "default_sets/cell_parameters/chen_2020.json");
cycling_protocol = load_cycling_protocol(; from_file_path = "default_sets/cycling_protocols/cc_discharge.json");

A loaded cell parameter set is a Dictionary-like object which comes with some additional handy functions. First, let's list the outermost keys of the cell parameters object.

In [75]:
keys(cell_parameters)

KeySet for a Dict{String, Any} with 6 entries. Keys:
  "Electrolyte"
  "Cell"
  "Metadata"
  "PositiveElectrode"
  "Separator"
  "NegativeElectrode"

Now we access the `Separator` key.

In [76]:
cell_parameters["Separator"]

Dict{String, Any} with 5 entries:
  "Description"          => "Ceramic-coated Polyolefin"
  "Density"              => 946
  "BruggemanCoefficient" => 1.5
  "Thickness"            => 1.2e-5
  "Porosity"             => 0.47

We have a flat list of parameters and values for the separator. In other cases, a key might nest other dictionaries, which can be accessed using the normal dictionary notation. Let's look at the active material parameters of the negative electrode.

In [77]:
cell_parameters["NegativeElectrode"]["ActiveMaterial"]

Dict{String, Any} with 16 entries:
  "ActivationEnergyOfDiffusion"       => 5000
  "NumberOfElectronsTransfered"       => 1
  "StoichiometricCoefficientAtSOC0"   => 0.0279
  "OpenCircuitPotential"              => "1.9793 * exp(-39.3631*(c/cmax)) + 0.2…
  "ReactionRateConstant"              => 6.716e-12
  "MassFraction"                      => 1.0
  "StoichiometricCoefficientAtSOC100" => 0.9014
  "ActivationEnergyOfReaction"        => 35000
  "MaximumConcentration"              => 33133.0
  "VolumetricSurfaceArea"             => 383959.0
  "Description"                       => "Graphite-SiOx"
  "DiffusionCoefficient"              => 3.3e-14
  "ParticleRadius"                    => 5.86e-6
  "Density"                           => 2260.0
  "ElectronicConductivity"            => 215
  "ChargeTransferCoefficient"         => 0.5

There are many parameters, nested into dictionaries. Often we are more interested in a specific subset of parameters. We can find a parameter with the `search_parameter` function. For example, let's see how area-related parameters are named:

In [78]:
search_parameter(cell_parameters, "area")


Parameters
------------------
[ "NegativeElectrode" ][ "ActiveMaterial" ][ "VolumetricSurfaceArea" ] => 383959.0
[ "PositiveElectrode" ][ "ActiveMaterial" ][ "VolumetricSurfaceArea" ] => 383959.0
[ "Cell" ][ "ElectrodeGeometricSurfaceArea" ] => 0.1027


Another way to view our parameters is by printing info about the parameter set.

In [79]:
print_info(cell_parameters)


PARAMETER OVERVIEW
Parameter                                                                                 Unit                Type                Value                         
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[ "Cell" ][ "Case" ]                                                                      N/A                 String              Cylindrical                   
[ "Cell" ][ "ElectrodeGeometricSurfaceArea" ]                                             m²                  Float64             0.1027                        
[ "Cell" ][ "Height" ]                                                                    m                   Float64             0.065                         
[ "Cell" ][ "InnerRadius" ]                                                               m                   Float64             0.002                         
[ "Cell" ][ "N

Parameters that take single numerical values (e.g. real, integers, booleans) can be directly modified.

In [80]:
cell_parameters["PositiveElectrode"]["Coating"]["Thickness"] = 8.2e-5

8.2e-5

Some parameters are described as functions or arrays, since the parameter value depends on other variables. For instance, the Open Circuit Potentials of the Active Materials depend on the lithium stoichiometry and temperature. When we're unsure about the type or meaning of a parameter, we can print information on individual parameters as well.

In [81]:
print_info("OpenCircuitPotential", view = "CellParameters")


----------------------------------------------------------------------------------------------------
🔋  Cell Parameter:  OpenCircuitPotential
----------------------------------------------------------------------------------------------------
    🔹 Name               OpenCircuitPotential
    🔹 Category           CellParameters
    🔹 Description        The open-circuit potential of the active material under a given intercalant stoichimetry and temperature.
    🔹 Type               String, Dict{String, Vector}, Real
    🔹 Unit               V
    🔹 Documentation      [visit](https://battmo.org/BattMo.jl/dev/manuals/user_guide/simulation_dependent_input)

    🔹 Ontology link      [visit](https://w3id.org/emmo/domain/electrochemistry#electrochemistry_9c657fdc_b9d3_4964_907c_f9a6e8c5f52b)




The cycling protocol parameters and the settings (model settings, simulation settings, solver settings) can be loaded, viewed and altered in the same way as the cell parameters. Let's load a default CCCV cycling protocol — we'll go into the settings later today.

In [82]:
cycling_protocol = load_cycling_protocol(; from_default_set = "cccv")
print_info(cycling_protocol)


PARAMETER OVERVIEW
Parameter                                                                                 Unit                Type                Value                         
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[ "CRate" ]                                                                               1                   Float64             1.0                           
[ "CurrentChangeLimit" ]                                                                  A·s⁻¹               Float64             0.0001                        
[ "DRate" ]                                                                               1                   Float64             1.0                           
[ "InitialControl" ]                                                                      N/A                 String              charging                      
[ "InitialStat

## 2 - Run a simulation ([docs](https://battmoteam.github.io/BattMo.jl/dev/manuals/user_guide/public_api))

Let's run a simple P2D simulation. We start again from the Chen 2020 cell parameter set and a constant current discharge cycling protocol.

In [83]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cycling_protocol = load_cycling_protocol(; from_default_set = "cc_discharge");

Next, we select the default Lithium-Ion Battery model. A model can be thought of as a mathematical implementation of the electrochemical and transport phenomena occurring in a real battery cell — a system of partial differential equations together with their parameters, constants and boundary conditions. The default setup below is a basic P2D model, without current collectors or SEI growth.

In [84]:
model = LithiumIonBattery();

✔️ Validation of ModelSettings passed: No issues found.
──────────────────────────────────────────────────


The `LithiumIonBattery` constructor validates the model settings in the background. If the model setup is valid, we can create a `Simulation` object by passing the model, cell parameters and cycling protocol. The `Simulation` object validates the parameters and settings too — each set is checked for being sensible and complete.

In [85]:
sim = Simulation(model, cell_parameters, cycling_protocol);

✔️ Validation of CellParameters passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of CyclingProtocol passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of SimulationSettings passed: No issues found.
──────────────────────────────────────────────────


When the `Simulation` object is valid we can solve it by passing it to `solve`. As Julia is a compiled language, the first time we run a simulation it will take some time to compile the functions and structs it encounters — a second run will be much faster.

In [86]:
output = solve(sim);

✔️ Validation of SolverSettings passed: No issues found.
──────────────────────────────────────────────────
Jutul: Simulating 2 hours, 12 minutes as 163 report steps


Progress   1%|█                                          |  ETA: 0:00:24

Progress   3%|██                                         |  ETA: 0:00:15

Progress   5%|███                                        |  ETA: 0:00:10

Progress   7%|████                                       |  ETA: 0:00:09

Progress  10%|█████                                      |  ETA: 0:00:08

Progress  13%|██████                                     |  ETA: 0:00:07

Progress  16%|███████                                    |  ETA: 0:00:06

Progress  19%|█████████                                  |  ETA: 0:00:05

Progress  22%|██████████                                 |  ETA: 0:00:05

Progress  26%|████████████                               |  ETA: 0:00:04

Progress  29%|█████████████                              |  ETA: 0:00:04

Progress  33%|███████████████                            |  ETA: 0:00:03

Progress  37%|████████████████                           |  ETA: 0:00:03

Progress  41%|██████████████████      

╭────────────────┬───────────┬───────────────┬──────────╮
│ Iteration type │  Avg/step │  Avg/ministep │    Total │
│                │ 146 steps │ 146 ministeps │ (wasted) │
├────────────────┼───────────┼───────────────┼──────────┤
│ Newton         │   2.32192 │       2.32192 │  339 (0) │
│ Linearization  │   3.32192 │       3.32192 │  485 (0) │
│ Linear solver  │   2.32192 │       2.32192 │  339 (0) │
│ Precond apply  │       0.0 │           0.0 │    0 (0) │
╰────────────────┴───────────┴───────────────┴──────────╯
╭───────────────┬─────────┬────────────┬────────╮
│ Timing type   │    Each │   Relative │  Total │
│               │      ms │ Percentage │      s │
├───────────────┼─────────┼────────────┼────────┤
│ Properties    │  0.7792 │     7.53 % │ 0.2642 │
│ Equations     │  1.9046 │    26.34 % │ 0.9237 │
│ Assembly      │  1.3660 │    18.89 % │ 0.6625 │
│ Linear solve  │  1.3460 │    13.01 % │ 0.4563 │
│ Linear setup  │  0.0000 │     0.00 % │ 0.0000 │
│ Precond apply │  0.0000 │ 

We can use built-in functions for quick plotting. The dashboard gives a quick overview of important output variables — `"contour"` shows position and time in one plot, `"line"` gives an interactive line plot with a time slider.

In [87]:
plot_dashboard(output; plot_type = "contour")

In [88]:
plot_dashboard(output; plot_type = "line")

Close all plotting windows before moving on.

In [89]:
GLMakie.closeall()

## 3 - Output ([docs](https://battmoteam.github.io/BattMo.jl/dev/tutorials/3_handle_outputs))

In BattMo.jl the output variables are divided into three categories:
- **time series**: all variables that depend on time.
- **states**: all the state variables, which can depend on time, axial position and radial position.
- **metrics**: the calculated cell metrics, dependent on the cycle index.

Let's simulate a couple of constant current constant voltage cycles to look into these.

In [90]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
cycling_protocol = load_cycling_protocol(; from_default_set = "cccv")

cycling_protocol["TotalNumberOfCycles"] = 10

model = LithiumIonBattery()
sim = Simulation(model, cell_parameters, cycling_protocol)

output = solve(sim)

plot_dashboard(output; plot_type = "simple")

✔️ Validation of ModelSettings passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of CellParameters passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of CyclingProtocol passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of SimulationSettings passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of SolverSettings passed: No issues found.
──────────────────────────────────────────────────
Jutul: Simulating 2 days, 2 hours as 3600 report steps


Progress   0%|█                                          |  ETA: 0:01:01

Progress   0%|█                                          |  ETA: 0:00:50

Progress   1%|█                                          |  ETA: 0:01:12

Progress   2%|█                                          |  ETA: 0:00:51

Progress   2%|█                                          |  ETA: 0:00:47

Progress   3%|██                                         |  ETA: 0:00:35

Progress   4%|██                                         |  ETA: 0:02:07

Progress   5%|███                                        |  ETA: 0:01:30

Progress   7%|███                                        |  ETA: 0:02:21

Progress   7%|████                                       |  ETA: 0:02:18

Progress   8%|████                                       |  ETA: 0:01:59

Progress   9%|████                                       |  ETA: 0:01:51

Progress  10%|█████                                      |  ETA: 0:01:38

Progress  11%|█████                   

╭────────────────┬────────────┬────────────────┬─────────────╮
│ Iteration type │   Avg/step │   Avg/ministep │       Total │
│                │ 2599 steps │ 2717 ministeps │    (wasted) │
├────────────────┼────────────┼────────────────┼─────────────┤
│ Newton         │    2.73451 │        2.61575 │ 7107 (1620) │
│ Linearization  │    3.77992 │        3.61575 │ 9824 (1701) │
│ Linear solver  │    2.73451 │        2.61575 │ 7107 (1620) │
│ Precond apply  │        0.0 │            0.0 │       0 (0) │
╰────────────────┴────────────┴────────────────┴─────────────╯
╭───────────────┬────────┬────────────┬─────────╮
│ Timing type   │   Each │   Relative │   Total │
│               │     ms │ Percentage │       s │
├───────────────┼────────┼────────────┼─────────┤
│ Properties    │ 0.0899 │     2.93 % │  0.6391 │
│ Equations     │ 0.2836 │    12.76 % │  2.7860 │
│ Assembly      │ 0.1221 │     5.49 % │  1.1998 │
│ Linear solve  │ 2.0548 │    66.87 % │ 14.6036 │
│ Linear setup  │ 0.0000 │     0.

Let's see which output variables are available.

In [91]:
print_info(output)


OUTPUT OVERVIEW
Variable                                                                                                 Unit                Shape                         
-----------------------------------------------------------------------------------------------------------------------------------------------------------

TIME_SERIES
Variable                                                                                                 Unit                Shape                         
-----------------------------------------------------------------------------------------------------------------------------------------------------------
[ "CumulativeCapacity" ]                                                                                 Ah                  (nTime,)                      
[ "Current" ]                                                                                            A                   (nTime,)                      
[ "CycleNumber" ]                 

We can see the variables are divided into the three categories described above. Let's retrieve some time series data: voltage, current and time.

In [92]:
time_series = output.time_series

t = time_series["Time"]
E = time_series["Voltage"]
I = time_series["Current"];

Let's also retrieve some state variables.

In [93]:
states = output.states

electrolyte_concentration = states["Electrolyte"]["Concentration"]
electrolyte_potential = states["Electrolyte"]["Potential"];

We can print more information on an individual variable, for example to look into the difference between the electrode particle concentration and the surface concentration.

In [94]:
print_info("NegativeElectrodeActiveMaterialParticleConcentration")


----------------------------------------------------------------------------------------------------
📈  Output Variable:  NegativeElectrodeActiveMaterialParticleConcentration
----------------------------------------------------------------------------------------------------
    🔹 Name               NegativeElectrodeActiveMaterialParticleConcentration
    🔹 Category           OutputVariable
    🔹 Description        Radial distribution of lithium concentration in negative electrode particles.
    🔹 Type               Vector{Real}
    🔹 Shape              (nTime, nPosition, nRadius)
    🔹 Unit               mol·L⁻¹



In [95]:
print_info("NegativeElectrodeActiveMaterialSurfaceConcentration")


----------------------------------------------------------------------------------------------------
📈  Output Variable:  NegativeElectrodeActiveMaterialSurfaceConcentration
----------------------------------------------------------------------------------------------------
    🔹 Name               NegativeElectrodeActiveMaterialSurfaceConcentration
    🔹 Category           OutputVariable
    🔹 Description        Concentration of lithium ions at the surface of negative electrode particles.
    🔹 Type               Vector{Real}
    🔹 Shape              (nTime, nPosition)
    🔹 Unit               mol·L⁻¹



We can also retrieve metrics from the output, like the discharge capacity and round trip efficiency per cycle.

In [96]:
metrics = output.metrics

discharge_capacity = metrics["DischargeCapacity"]
round_trip_efficiency = metrics["RoundTripEfficiency"]
cycle_index = metrics["CycleIndex"];

Let's plot the discharge capacity and round trip efficiency against cycle index.

In [97]:
f = Figure(size = (1000, 400))

ax = Axis(f[1, 1], title = "Round trip efficiency", xlabel = "Cycle number / -", ylabel = "Efficiency / %")
scatterlines!(ax, cycle_index, round_trip_efficiency; linewidth = 4)

ax = Axis(f[2, 1], title = "Discharge capacity", xlabel = "Cycle number / -", ylabel = "Capacity / Ah")
scatterlines!(ax, cycle_index, discharge_capacity; linewidth = 4)

f

In [98]:
GLMakie.closeall()

## Assignment — find the cliff

So far you've only run the `cc_discharge` protocol at its default rate. Real cells don't perform equally well at every discharge rate — capacity holds up for a while, then drops off a "cliff" at high rates, and beyond some point the solver may not converge at all.

Your task: sweep the `DRate` of the `cc_discharge` protocol from low to high (e.g. 0.2, 0.5, 1, 2, 3, 4, 5, 6 C), record the discharge capacity at each rate, and plot capacity vs. `DRate`. Wrap the simulation in a `try`/`catch` — at high rates the solver may fail outright rather than return a sensible result, so push `0.0` in that case.

In [103]:
cell_parameters = load_cell_parameters(; from_default_set = "chen_2020")
discharge_protocol = load_cycling_protocol(; from_default_set = "cc_discharge")
model = LithiumIonBattery()

d_rates = [0.2, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
capacities = Float64[]

for d_rate in d_rates
    discharge_protocol["DRate"] = d_rate

    # --- build and solve the simulation, then push the discharge capacity (or 0.0 on failure) to `capacities`: push!(capacities, capacity) ---

    sim = Simulation(model, cell_parameters, discharge_protocol)
    output = solve(sim)
    
    push!(capacities, output.metrics["DischargeCapacity"][1])

    # --------------------------------------------------------------------------------------------------------
end

✔️ Validation of ModelSettings passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of CellParameters passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of CyclingProtocol passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of SimulationSettings passed: No issues found.
──────────────────────────────────────────────────
✔️ Validation of SolverSettings passed: No issues found.
──────────────────────────────────────────────────
Jutul: Simulating 5 hours, 30 minutes as 401 report steps


Progress   3%|██                                         |  ETA: 0:00:04

Progress  12%|██████                                     |  ETA: 0:00:02

Progress  21%|██████████                                 |  ETA: 0:00:01

Progress  28%|████████████                               |  ETA: 0:00:01

Progress  38%|█████████████████                          |  ETA: 0:00:01

Progress  46%|████████████████████                       |  ETA: 0:00:01

Progress  54%|████████████████████████                   |  ETA: 0:00:01

Progress  62%|███████████████████████████                |  ETA: 0:00:01

Progress  70%|██████████████████████████████             |  ETA: 0:00:00

Progress  77%|█████████████████████████████████          |  ETA: 0:00:00

Progress  84%|█████████████████████████████████████      |  ETA: 0:00:00

Progress 100%|███████████████████████████████████████████| Time: 0:00:01


╭────────────────┬───────────┬───────────────┬──────────╮
│ Iteration type │  Avg/step │  Avg/ministep │    Total │
│                │ 360 steps │ 360 ministeps │ (wasted) │
├────────────────┼───────────┼───────────────┼──────────┤
│ Newton         │   2.06667 │       2.06667 │  744 (0) │
│ Linearization  │   3.06667 │       3.06667 │ 1104 (0) │
│ Linear solver  │   2.06667 │       2.06667 │  744 (0) │
│ Precond apply  │       0.0 │           0.0 │    0 (0) │
╰────────────────┴───────────┴───────────────┴──────────╯
╭───────────────┬────────┬────────────┬────────╮
│ Timing type   │   Each │   Relative │  Total │
│               │     ms │ Percentage │      s │
├───────────────┼────────┼────────────┼────────┤
│ Properties    │ 0.0767 │     4.19 % │ 0.0571 │
│ Equations     │ 0.2777 │    22.49 % │ 0.3066 │
│ Assembly      │ 0.1014 │     8.21 % │ 0.1119 │
│ Linear solve  │ 0.8215 │    44.83 % │ 0.6112 │
│ Linear setup  │ 0.0000 │     0.00 % │ 0.0000 │
│ Precond apply │ 0.0000 │     0.00 %

Progress  13%|██████                                     |  ETA: 0:00:01

Progress  32%|██████████████                             |  ETA: 0:00:00

Progress  49%|██████████████████████                     |  ETA: 0:00:00

Progress  67%|█████████████████████████████              |  ETA: 0:00:00

Progress  84%|█████████████████████████████████████      |  ETA: 0:00:00

Progress 100%|███████████████████████████████████████████| Time: 0:00:00


╭────────────────┬───────────┬───────────────┬──────────╮
│ Iteration type │  Avg/step │  Avg/ministep │    Total │
│                │ 146 steps │ 146 ministeps │ (wasted) │
├────────────────┼───────────┼───────────────┼──────────┤
│ Newton         │   2.32192 │       2.32192 │  339 (0) │
│ Linearization  │   3.32192 │       3.32192 │  485 (0) │
│ Linear solver  │   2.32192 │       2.32192 │  339 (0) │
│ Precond apply  │       0.0 │           0.0 │    0 (0) │
╰────────────────┴───────────┴───────────────┴──────────╯
╭───────────────┬────────┬────────────┬──────────╮
│ Timing type   │   Each │   Relative │    Total │
│               │     ms │ Percentage │       ms │
├───────────────┼────────┼────────────┼──────────┤
│ Properties    │ 0.0963 │     4.92 % │  32.6488 │
│ Equations     │ 0.4160 │    30.39 % │ 201.7434 │
│ Assembly      │ 0.1113 │     8.13 % │  53.9628 │
│ Linear solve  │ 0.7008 │    35.79 % │ 237.5558 │
│ Linear setup  │ 0.0000 │     0.00 % │   0.0000 │
│ Precond apply │ 0

...

╭────────────────┬──────────┬──────────────┬──────────╮
│ Iteration type │ Avg/step │ Avg/ministep │    Total │
│                │ 15 steps │ 49 ministeps │ (wasted) │
├────────────────┼──────────┼──────────────┼──────────┤
│ Newton         │  13.8667 │       4.2449 │ 208 (66) │
│ Linearization  │  15.6667 │      4.79592 │ 235 (67) │
│ Linear solver  │     12.4 │      3.79592 │ 186 (44) │
│ Precond apply  │      0.0 │          0.0 │    0 (0) │
╰────────────────┴──────────┴──────────────┴──────────╯
╭───────────────┬────────┬────────────┬──────────╮
│ Timing type   │   Each │   Relative │    Total │
│               │     ms │ Percentage │       ms │
├───────────────┼────────┼────────────┼──────────┤
│ Properties    │ 0.0662 │     4.67 % │  13.7639 │
│ Equations     │ 0.2758 │    21.97 % │  64.8059 │
│ Assembly      │ 0.2682 │    21.37 % │  63.0307 │
│ Linear solve  │ 0.4032 │    28.43 % │  83.8673 │
│ Linear setup  │ 0.0000 │     0.00 % │   0.0000 │
│ Precond apply │ 0.0000 │     0.00 %

Progress  36%|████████████████                           |  ETA: 0:00:01

Progress 100%|███████████████████████████████████████████| Time: 0:00:00


╭────────────────┬──────────┬──────────────┬──────────╮
│ Iteration type │ Avg/step │ Avg/ministep │    Total │
│                │  8 steps │ 72 ministeps │ (wasted) │
├────────────────┼──────────┼──────────────┼──────────┤
│ Newton         │   29.625 │      3.29167 │ 237 (90) │
│ Linearization  │   33.125 │      3.68056 │ 265 (90) │
│ Linear solver  │   24.125 │      2.68056 │ 193 (46) │
│ Precond apply  │      0.0 │          0.0 │    0 (0) │
╰────────────────┴──────────┴──────────────┴──────────╯
╭───────────────┬────────┬────────────┬──────────╮
│ Timing type   │   Each │   Relative │    Total │
│               │     ms │ Percentage │       ms │
├───────────────┼────────┼────────────┼──────────┤
│ Properties    │ 0.0594 │     4.15 % │  14.0728 │
│ Equations     │ 0.2750 │    21.50 % │  72.8810 │
│ Assembly      │ 0.0982 │     7.68 % │  26.0279 │
│ Linear solve  │ 0.5137 │    35.92 % │ 121.7384 │
│ Linear setup  │ 0.0000 │     0.00 % │   0.0000 │
│ Precond apply │ 0.0000 │     0.00 %

Progress  32%|██████████████                             |  ETA: 0:00:01

Progress 100%|███████████████████████████████████████████| Time: 0:00:00


╭────────────────┬──────────┬──────────────┬───────────╮
│ Iteration type │ Avg/step │ Avg/ministep │     Total │
│                │  5 steps │ 72 ministeps │  (wasted) │
├────────────────┼──────────┼──────────────┼───────────┤
│ Newton         │     43.2 │          3.0 │ 216 (102) │
│ Linearization  │     47.6 │      3.30556 │ 238 (102) │
│ Linear solver  │     33.4 │      2.31944 │  167 (53) │
│ Precond apply  │      0.0 │          0.0 │     0 (0) │
╰────────────────┴──────────┴──────────────┴───────────╯
╭───────────────┬────────┬────────────┬──────────╮
│ Timing type   │   Each │   Relative │    Total │
│               │     ms │ Percentage │       ms │
├───────────────┼────────┼────────────┼──────────┤
│ Properties    │ 0.0567 │     3.96 % │  12.2566 │
│ Equations     │ 0.2678 │    20.60 % │  63.7399 │
│ Assembly      │ 0.1009 │     7.76 % │  24.0070 │
│ Linear solve  │ 0.5931 │    41.39 % │ 128.1113 │
│ Linear setup  │ 0.0000 │     0.00 % │   0.0000 │
│ Precond apply │ 0.0000 │  

Now plot capacity vs. `DRate` to see where the cliff is.

In [104]:
f = Figure(size = (700, 400))
ax = Axis(f[1, 1], title = "Rate capability", xlabel = "D-rate / C", ylabel = "Discharge capacity / Ah")
scatterlines!(ax, d_rates, capacities; linewidth = 4)
f

In [101]:
GLMakie.closeall()

At roughly what rate does the cliff start? Keep this in mind — this afternoon's case-solving brief has a requirement that's exactly this trade-off (keeping at least 90% of capacity at 2C compared to 0.5C).